In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("GPU Count:", torch.cuda.device_count())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU Found")


In [ ]:
from unsloth import FastLanguageModel
import torch
import numpy as np

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42
)

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("2022_Patient_level_prompts.csv")

dataset = Dataset.from_pandas(df)
dataset = dataset.shuffle(seed=42)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")  

def format_conversations(example):
    conversation = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]}
    ]


    formatted_text = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=False
    )

    return {"text": formatted_text}

dataset = dataset.map(format_conversations)


In [ ]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args =TrainingArguments(
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,
        warmup_ratio=0.1,
        learning_rate=2e-5,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        save_strategy="no",  
        output_dir="llama-3.2-3b-instruct-2e-5-Checkpoints"
    )

In [ ]:
from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False)


trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    data_collator = data_collator,   
    dataset_num_proc=10,
    packing=True,
    args=training_args
)



In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n"
)


In [ ]:
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("llama-3.2-3b_finetuned") 
tokenizer.save_pretrained("llama-3.2-3b_finetuned")